## Step 7: Transfer Learning using MobileNetV2

In this step, transfer learning is applied to improve pneumonia detection performance. A pretrained MobileNetV2 model is used as a feature extractor, leveraging knowledge learned from large-scale image datasets.

Unlike training from scratch, transfer learning enables the model to capture complex visual patterns with limited data, significantly improving generalization.

This approach addresses the limitations observed in previous models and is expected to achieve higher accuracy and robustness.


In [1]:
# Import libraries

import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

2026-05-03 09:16:42.092372: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777799802.388152      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777799802.475853      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777799803.146508      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777799803.146845      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777799803.146853      57 computation_placer.cc:177] computation placer alr

In [2]:
#  Load dataset

df = pd.read_csv("/kaggle/input/datasets/sairasagnak/balanced-data-rsna/balanced_data.csv")
print(df.shape)

(12024, 8)


In [4]:
#  Fix paths

DATA_DIR = "/kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset"
IMAGE_DIR = os.path.join(DATA_DIR, "Training", "Images")

df['image_path'] = df['patientId'].apply(lambda x: os.path.join(IMAGE_DIR, x + ".png"))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(df.shape)

(12024, 8)


In [6]:
#  Preprocessing (IMPORTANT — 3 channels for pretrained model)

IMG_SIZE = 128

def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

In [7]:
# Prepare dataset

df = df.sample(3000, random_state=42)

images = []
labels = []

for _, row in df.iterrows():
    img = preprocess_image(row['image_path'])
    images.append(img)
    labels.append(row['Target'])

X = np.array(images)
y = np.array(labels)

print(X.shape, y.shape)

(3000, 128, 128, 3) (3000,)


In [8]:
#  Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
#  Load pretrained model

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

2026-05-03 09:22:23.667495: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
# Build model

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [12]:
#  Compile

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
#  Train

history = model.fit(
    X_train, y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 29s 320ms/step - accuracy: 0.6689 - loss: 0.6309 - val_accuracy: 0.7450 - val_loss: 0.5235
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 21s 287ms/step - accuracy: 0.7658 - loss: 0.4774 - val_accuracy: 0.7483 - val_loss: 0.5186
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 21s 280ms/step - accuracy: 0.7783 - loss: 0.4732 - val_accuracy: 0.7600 - val_loss: 0.5204
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 21s 286ms/step - accuracy: 0.8169 - loss: 0.4237 - val_accuracy: 0.7517 - val_loss: 0.5191
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 22s 288ms/step - accuracy: 0.7892 - loss: 0.4318 - val_accuracy: 0.7533 - val_loss: 0.5417


In [14]:
#  Evaluate

loss, accuracy = model.evaluate(X_test, y_test)

print("Transfer Learning Accuracy:", accuracy)

19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 233ms/step - accuracy: 0.7644 - loss: 0.5059
Transfer Learning Accuracy: 0.753333330154419


## Step 7: Observations

* The transfer learning model achieved an accuracy of approximately 75%, comparable to the baseline MLP model.
* Freezing the pretrained MobileNetV2 layers limited the model’s ability to adapt to domain-specific features present in chest X-ray images.
* The model showed moderate improvement in training accuracy, but validation performance remained relatively stable, indicating limited generalization gains.
* The relatively small dataset and absence of data augmentation constrained the effectiveness of transfer learning.
* These results highlight that transfer learning requires fine-tuning and domain adaptation to fully leverage pretrained features.

This step demonstrates that pretrained models alone are not sufficient; proper fine-tuning and training strategies are essential for achieving optimal performance.
